# Read me

This notebook uses a NearestNeighbors model with Euclidean distance to recommend 20 tracks based on the profile of 10 input tracks. The profile is built by calculating the mean of audio features of the input tracks. 

9 features → k-NN → 20 recommandations

In [1]:
import os
from pathlib import Path

import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import joblib

import app.spotify.spotify as spo
from app.preprocessing.preprocessing import preprocessing


In [30]:
# Set working directory to project root
project_root = Path.cwd().parent
os.chdir(project_root)

project_root

PosixPath('/Users/tancrede')

# Load dataset

In [2]:
df = pd.read_csv("../raw_data/spotify-tracks-dataset-detailed.csv")
print(f'Loaded {len(df)} rows')

Loaded 114000 rows


# Preprocessing

In [3]:
X_scaled, df_metadata, scaler, features = preprocessing(df)


audio_features = features

print(f'Loaded {len(X_scaled)} rows')

Loaded 80175 rows


In [5]:
# save scaler
joblib.dump(scaler, "../artifacts/models/baseline/baseline_scaler.joblib")

['../artifacts/models/baseline/baseline_scaler.joblib']

# Cleaning dataset
**Deprecated: use preprocessing.py instead**

## Removing Comedy genre

In [10]:
len_before = len(df)
df = df[df["track_genre"] != "comedy"].reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} comedy tracks. Left {len_after} rows')

Removed 1000 comedy tracks. Left 113000 rows


## Drop duplicates

In [11]:
len_before = len(df)
df = df.drop_duplicates(subset=["track_id"]).drop_duplicates(subset=["artists", "track_name"]).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} duplicates. Left {len_after} rows')

Removed 32642 duplicates. Left 80358 rows


## Drop missing values

In [12]:
len_before = len(df)
df = df.dropna(subset=['album_name']).dropna(subset=['track_name']).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} null values. Left {len(df)} rows')

Removed 1 null values. Left 80357 rows


In [13]:
# save catalog
df.to_parquet("artifacts/catalog_clean.parquet", index=False)

## Scaling

In [14]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "valence",
    "tempo"
]

scaler = StandardScaler().set_output(transform="pandas")

X = df[audio_features]

X_scaled = scaler.fit_transform(X)

In [15]:
# save scaler
joblib.dump(scaler, "artifacts/models/baseline/baseline_scaler.joblib")

['artifacts/models/baseline/baseline_scaler.joblib']

In [16]:
# save scaled catalog
X_scaled.to_parquet("artifacts/models/baseline/catalog_scaled.parquet", index=False)

In [17]:
X_scaled.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
0,0.655383,-0.667949,0.343899,0.742394,-0.865010,-0.561572,0.953320,-1.147548
1,-0.779302,-1.805605,-1.632471,-0.052910,1.771528,-0.561558,-0.744136,-1.494515
2,-0.678426,-1.061308,-0.219109,-0.298536,-0.339358,-0.561575,-1.301114,-1.533011
3,-1.642355,-2.215933,-1.873653,-0.529854,1.715356,-0.561362,-1.213967,1.974189
4,0.330337,-0.737365,-0.209123,-0.335499,0.426356,-0.561575,-1.123032,-0.081759


In [18]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Model 1
- NearestNeighbors with Euclidean distance
- Input tracks: Average acoustic profile (Centroid k-NN)

## Train

In [6]:
# NearestNeighbors with Euclidean distance
model = NearestNeighbors(metric="euclidean")

model.fit(X_scaled)

,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [8]:
# Save model

joblib.dump(model, "../artifacts/models/baseline/baseline_model.joblib")

['../artifacts/models/baseline/baseline_model.joblib']

## Input tracks

In [16]:
# Build an input track set of 10 tracks
input_tracks = X_scaled.sample(n=10)

In [17]:
# Print input track names
input_indices = input_tracks.index.to_list()
df.iloc[input_indices][['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
47277,31puVs5TMVVSIPuteoN2X0,Seiko Matsuda,ユートピア,ピーチ・シャーベット,j-idol
47803,4QkKRtOPA0mU55JoyVqtQV,Sertab Erener,İyiliğe Ninniler,Çamlıbelden Çıktım Yayan,j-pop
43849,12sjKNSed91fDBZgxle2P2,Nucleya;Sivamani;Chinna Ponnu,Bass Rani,Chennai Bass,indie-pop
19086,22XbNq1dFj7ASyx5f1anYC,Two Without Hats,"Micmac's Greatest Freestyle Hits!, Vol. 1",Try Yazz,deep-house
12454,2gWQbYlLaC04j3teJnFKr9,CoComelon,Halloween Party,Pumpkin Carving,children
12831,7LEMarOitZtIskRnm4CRRB,VeggieTales,VeggieTunes,I Can Be Your Friend,children
62407,1orVKbp6vqtfAPOmvRofVq,Steve Miller Band,Fly Like An Eagle,Rock'n Me,psych-rock
40866,0pmbq9vYE7bTLs14YoD7eL,Pete Bernhard,Straight Line,Mr. Mudd and Mr. Gold,honky-tonk
16263,24tNhacN8vdN8OqkR7LcNm,Lady Gaga,Workout Hits 2023,Poker Face,dance
3090,1ulB5740qjdSila4Vjnb7Z,Novo Amor,"Cannot Be, Whatsoever",Birdcage,ambient


In [37]:
token = spo.get_access_token()
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[input_indices]['track_id'])
spo.display_player(playlist_id)

In [38]:
# Calculate input profile
input_profile = input_tracks.mean(axis=0).to_frame().T

input_profile

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.562913,-0.074439,-0.031101,0.008258,0.216107,-0.011646,-0.393387,0.543733,-0.206498


## Recommend

In [39]:
# Find 20 neighbors
distances, indices = model.kneighbors(input_profile, n_neighbors=20)

# The model returns an array of indices
# Return the tracks from the original dataframe
recommendations = df.iloc[indices[0]]


In [20]:
# Print recommendations profile
recommendations_indices = recommendations.index.to_list()
recommendations_tracks = X_scaled.iloc[recommendations_indices]

recommendations_profile = recommendations_tracks.mean(axis=0).to_frame().T

recommendations_profile

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,-0.087739,0.341673,0.504991,-0.18085,-0.582112,-0.500308,-0.411382,0.028623,0.77791


In [21]:
playlist_id = spo.create_playlist(token)
spo.add_tracks_to_playlist(token, playlist_id, df.iloc[recommendations_indices]['track_id'])
spo.display_player(playlist_id)

In [22]:
recommendations[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
25048,6Dg3ArDiJcseYoeeTZZXK6,San Holo,All The Highs,All The Highs,electronic
51830,73pv8Je9LeooK0UgE9mXRY,Havoc Mathan;Havoc Naven,Best of Havoc Brothers,Pirantha Naal,malay
37078,7axJf0rBRxyTk397LbJpTt,Pitty;Drik Barbosa;WEKS,Casulo,Diamante,hard-rock
50606,3CpD9lnEHelRkBtsdktwHy,Luis Fonsi,Ya huele a buñuelos,Let It Snow,latin
60440,3fVnlF4pGqWI9flVENcT28,Taylor Swift,1989,Wildest Dreams,pop
31329,2rcELfYmKTFtit6Gv1ZEqF,Canção & Louvor,Eu Sou Mistério,Eu Sou Mistério,gospel
66705,7jahcsevTvePvdduUFVzpq,Aries,WELCOME HOME,CAROUSEL,sad
39276,5oYRR7SsbqnpXKaXAwbH5m,A.N.I.M.A.L.;Chuck Jonhson,6 (Animal),Buscando llegar hasta el sol,heavy-metal
31439,1wlAJE1E8Dom0LwgAImVg9,Lauriete,João Viu,João Viu,gospel
51769,4QzJOTh94tmzsR1lV6mfML,Sajeer Koppam,"Dilse, Vol. 2",Vannu ninte,malay


# Alternative models
- Model: NearestNeighbors with Manhattan distance
- Input tracks: average of multimodal preferences (Multi-seed k-NN)

['../app/models/baseline_knn_model.joblib']